<a href="https://colab.research.google.com/github/VEMULAVISHNUVARDHAN4076/Natural-Language-Processing/blob/main/2403A54076_NLP_LAB_Assignment_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
!pip install gensim

In [24]:
# Numerical operations
import numpy as np

# Keras modules
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Conv1D
from tensorflow.keras.layers import GlobalMaxPooling1D, Dense, Concatenate
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Gensim for Word2Vec
from gensim.models import Word2Vec

In [25]:
texts = [
    "this movie is amazing",
    "i really love this film",
    "fantastic story and acting",
    "what a great experience",
    "absolutely wonderful movie",

    "this movie is bad",
    "i really hate this film",
    "terrible acting and story",
    "worst experience ever",
    "completely boring movie"
]

labels = [
    1, 1, 1, 1, 1,   # Positive
    0, 0, 0, 0, 0    # Negative
]

In [26]:
tokenizer = Tokenizer()

# Learn vocabulary
tokenizer.fit_on_texts(texts)

# Convert sentences into sequences
sequences = tokenizer.texts_to_sequences(texts)

# Vocabulary dictionary
word_index = tokenizer.word_index

print(word_index)

{'this': 1, 'movie': 2, 'is': 3, 'i': 4, 'really': 5, 'film': 6, 'story': 7, 'and': 8, 'acting': 9, 'experience': 10, 'amazing': 11, 'love': 12, 'fantastic': 13, 'what': 14, 'a': 15, 'great': 16, 'absolutely': 17, 'wonderful': 18, 'bad': 19, 'hate': 20, 'terrible': 21, 'worst': 22, 'ever': 23, 'completely': 24, 'boring': 25}


In [27]:
print(sequences)

[[1, 2, 3, 11], [4, 5, 12, 1, 6], [13, 7, 8, 9], [14, 15, 16, 10], [17, 18, 2], [1, 2, 3, 19], [4, 5, 20, 1, 6], [21, 9, 8, 7], [22, 10, 23], [24, 25, 2]]


In [28]:
max_length = 6

X = pad_sequences(sequences, maxlen=max_length)

y = np.array(labels)

In [29]:
X

array([[ 0,  0,  1,  2,  3, 11],
       [ 0,  4,  5, 12,  1,  6],
       [ 0,  0, 13,  7,  8,  9],
       [ 0,  0, 14, 15, 16, 10],
       [ 0,  0,  0, 17, 18,  2],
       [ 0,  0,  1,  2,  3, 19],
       [ 0,  4,  5, 20,  1,  6],
       [ 0,  0, 21,  9,  8,  7],
       [ 0,  0,  0, 22, 10, 23],
       [ 0,  0,  0, 24, 25,  2]], dtype=int32)

In [30]:
sentences = [text.split() for text in texts]

# Train Word2Vec
w2v_model = Word2Vec(
    sentences,
    vector_size=50,
    window=3,
    min_count=1
)

In [31]:
vocab_size = len(word_index) + 1
embedding_dim = 50

embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, index in word_index.items():
    if word in w2v_model.wv:
        embedding_matrix[index] = w2v_model.wv[word]

In [32]:
embedding_matrix[1]

array([-0.01632186,  0.00899384, -0.00827396,  0.00165139,  0.0170025 ,
       -0.00893005,  0.00903778, -0.01357326, -0.00710123,  0.01879604,
       -0.0031543 ,  0.00063798, -0.00828458, -0.0153624 , -0.00301697,
        0.00494605, -0.00176969,  0.0110757 , -0.00549004,  0.00452046,
        0.01091541,  0.01669304, -0.00290107, -0.01842383,  0.00874634,
        0.00114724,  0.01488473, -0.00162445, -0.00527902, -0.01750613,
       -0.00170662,  0.00565221,  0.01080266,  0.01410515, -0.01140646,
        0.00371918,  0.01218152, -0.0095973 , -0.00620937,  0.01360143,
        0.00326828,  0.00037596,  0.00694588,  0.00043366,  0.01924119,
        0.01012958, -0.01783511, -0.01408798,  0.00180418,  0.01278653])

In [33]:
input_layer = Input(shape=(max_length,))

In [34]:
embedding_layer = Embedding(
    input_dim=vocab_size,
    output_dim=embedding_dim,
    weights=[embedding_matrix],  # Word2Vec weights
    input_length=max_length,
    trainable=False               # keep embeddings fixed
)(input_layer)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [35]:
conv1 = Conv1D(filters=100, kernel_size=3, activation='relu')(embedding_layer)

conv2 = Conv1D(filters=100, kernel_size=4, activation='relu')(embedding_layer)

conv3 = Conv1D(filters=100, kernel_size=5, activation='relu')(embedding_layer)

In [36]:
pool1 = GlobalMaxPooling1D()(conv1)
pool2 = GlobalMaxPooling1D()(conv2)
pool3 = GlobalMaxPooling1D()(conv3)

In [38]:
merged = Concatenate()([pool1, pool2, pool3])

In [40]:
dense = Dense(10, activation='relu')(merged)

In [41]:
output = Dense(1, activation='sigmoid')(dense)

In [42]:
model = Model(inputs=input_layer, outputs=output)

In [43]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [44]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 6)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 6, 50)     │      1,300 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 4, 100)    │     15,100 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 3, 100)    │     20,100 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 2, 100)    │     25,100 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 100)       │          0 │ conv1d[0][0]      │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 100)       │          0 │ conv1d_1[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 100)       │          0 │ conv1d_2[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 300)       │          0 │ global_max_pooli… │
│ (Concatenate)       │                   │            │ global_max_pooli… │
│                     │                   │            │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 10)        │      3,010 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 1)         │         11 │ dense_1[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 64,621 (252.43 KB)

 Trainable params: 63,321 (247.35 KB)

 Non-trainable params: 1,300 (5.08 KB)

In [45]:
model.fit(
    X,
    y,
    epochs=10,
    batch_size=2
)

Epoch 1/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.5000 - loss: 0.6953
Epoch 2/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8000 - loss: 0.6890 
Epoch 3/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8000 - loss: 0.6837 
Epoch 4/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8000 - loss: 0.6765
Epoch 5/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.9000 - loss: 0.6698
Epoch 6/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9000 - loss: 0.6625
Epoch 7/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.8000 - loss: 0.6535
Epoch 8/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8000 - loss: 0.6426
Epoch 9/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 1.0000 - loss: 0.6311
Epoch 10/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 1.0000 - loss: 0.6176


In [46]:
test_text = ["this movie is amazing"]

seq = tokenizer.texts_to_sequences(test_text)

pad = pad_sequences(seq, maxlen=max_length)

prediction = model.predict(pad)

print(prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 224ms/step
[[0.50464064]]
